# Solar Filament V7 — CPU Native-Tile Instance Discovery

**Purpose:** A genuinely new experiment after repeated 0.31 public PQ. Reuse the user's trained YOLO11n-seg checkpoint and compare **its identical original full-disk V4 postprocessing** against **additional inference on native-resolution overlapping solar-image tiles**. This is not another threshold-only optimization and contains no encoded test predictions.

**CPU-only, checkpoint-first, independently reproducible:** Uses only the official challenge train polygons for tuning/audit, official test JPEG pixels for inference, and the attached user's YOLO checkpoint. Tile prediction JSON is cached with SHA256 provenance for restarting across Kaggle sessions. Local PQ is a **proxy**, not guaranteed to match the organizer's final evaluator.

**Scientific decision:** If native-tile proposals do not improve PQ by a meaningful amount across the full original grouped tuning split, the notebook writes a baseline submission instead of promoting them. The previous audit has been seen and is labelled a *regression check*, not a pristine holdout.

**Run modes:** `TUNE` (recommended first) generates tile predictions for 30 tuning images and identifies promising methods. `CONFIRM` reevaluates finalists across all 97 original tuning images and runs the previously seen 97-image audit comparison. `SUBMIT` exports the selected candidate. `ALL` performs every stage but is substantially more CPU inference. Maintain the same output dataset between Kaggle sessions and attach it using `PREVIOUS_V7_ROOT`.

Official competition overview and evaluator: https://www.kaggle.com/competitions/filament-segmentation-2026 and https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook.

In [ ]:
import sys, os, gc, json, hashlib, random, time, subprocess, importlib.util
from pathlib import Path
from collections import defaultdict
from itertools import product
import numpy as np, pandas as pd, cv2
from tqdm.auto import tqdm

# ========== Configuration ==========
MODE = 'TUNE'                 # TUNE | CONFIRM | SUBMIT | ALL
SEED = 2026
DATA_ROOT = Path('/kaggle/input/datasets/lighittha/solar3k/MAGFiLO_1.0_Kaggle_2026')
SOURCE_V4_ROOT = Path('/kaggle/input/datasets/lighittha/solarouts/solar_cpu_v4')
CHECKPOINT_OVERRIDE = None   # full path to your trained yolo11n-seg best.pt, if different
PREVIOUS_V7_ROOT = None     # optional attached previous V7 output folder (contains v7_config.json, tiles/, global/)
OUT = Path('/kaggle/working/solar_cpu_v7') if Path('/kaggle/working').exists() else Path('./solar_cpu_v7')
OUT.mkdir(parents=True,exist_ok=True)
DEVICE='cpu'

# Identical old global inference contract from V4/V6
GLOBAL_IMGSZ=1024; RAW_CONF=.10; RAW_IOU=.82; MAX_DET=65
# This changes detector input pixels: 4 native 1280x1280 overlapping views of each 2048x2048 observation.
TILE_SIDE=1280; TILE_IMGSZ=1024
TILES='FOUR'     # FOUR or FIVE (adds center view)
TUNE_SAMPLE=30   # split-independent sampling from the original V4 tune group
CONFIRM_TOP=8
AUDIT_PREVIOUSLY_SEEN=True
MIN_GAIN_TO_PROMOTE=.015  # on complete tune group; avoid chasing 0.002 improvements
VERIFY_GAINS_ON_PREVIOUS_AUDIT=True   # regression check only, not pristine evaluation
AUTO_INSTALL=True

random.seed(SEED);np.random.seed(SEED)
cv2.setNumThreads(0)
for module,pkg in [('ultralytics','ultralytics>=8.3,<9'),('pycocotools','pycocotools')]:
    if importlib.util.find_spec(module) is None:
        if not AUTO_INSTALL:raise ImportError(pkg)
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import torch,ultralytics
from ultralytics import YOLO
from pycocotools import mask as coco
from IPython.display import display
# Keep CPU execution manageable on shared Kaggle kernels
if hasattr(torch,'set_num_threads'):torch.set_num_threads(min(4,os.cpu_count() or 2))
print('Mode:',MODE,'device:',DEVICE,'torch:',torch.__version__,'ultralytics:',ultralytics.__version__)

## 1. Data and checkpoint integrity

Preserve V4's **exact original acquisition-day grouping**. No new split is constructed, because the existing weights were trained using that split. Every test observation is disjoint from labeled training observations. Checkpoint SHA256 ties proposal caches to their source weights.

In [ ]:
if not (DATA_ROOT/'train'/'MAGFiLO_1.0_Annotations_kaggle2026_train.json').exists():
    candidates=list(Path('/kaggle/input').rglob('MAGFiLO_1.0_Annotations_kaggle2026_train.json')) if Path('/kaggle/input').exists() else []
    if len(candidates)==1:DATA_ROOT=candidates[0].parent.parent
    else:raise FileNotFoundError('Attach official MAGFiLO competition dataset: '+str(candidates[:8]))
assert (SOURCE_V4_ROOT/'split_manifest.json').exists(), 'Attach your saved solar_cpu_v4 folder'
SPLIT=json.loads((SOURCE_V4_ROOT/'split_manifest.json').read_text())
assert set(SPLIT)=={'train','tune','audit'}
ANNS=json.loads((DATA_ROOT/'train'/'MAGFiLO_1.0_Annotations_kaggle2026_train.json').read_text())
TRAIN={p.stem:p for p in (DATA_ROOT/'train'/'train_images').iterdir() if p.suffix.lower() in ('.jpg','.jpeg','.png')}
TEST={p.stem:p for p in (DATA_ROOT/'test'/'test_images').iterdir() if p.suffix.lower() in ('.jpg','.jpeg','.png')}
assert len(TEST)==180 and len(TRAIN)==707,(len(TEST),len(TRAIN))
AB=defaultdict(list)
for a in ANNS['annotations']:AB[str(a['image_id'])].append(a)
OBS=defaultdict(list)
for meta in ANNS['images']:
    s=Path(meta['file_name']).stem
    if s in TRAIN:OBS[s].append((str(meta['id']),AB[str(meta['id'])]))
assert set(OBS)==set(TRAIN)
assert set.union(*(set(SPLIT[s]) for s in SPLIT))==set(TRAIN)
assert len(set(SPLIT['train'])&set(SPLIT['tune']))==0
assert len(set(SPLIT['train'])&set(SPLIT['audit']))==0
assert len(set(SPLIT['tune'])&set(SPLIT['audit']))==0
for a in ('train','tune','audit'):
    for b in ('train','tune','audit'):
        if a!=b:assert not ({x[:8] for x in SPLIT[a]}&{x[:8] for x in SPLIT[b]})

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''):h.update(chunk)
    return h.hexdigest()
CKPT=Path(CHECKPOINT_OVERRIDE) if CHECKPOINT_OVERRIDE else SOURCE_V4_ROOT/'runs'/'yolo11n-seg'/'weights'/'best.pt'
assert CKPT.is_file(),CKPT
MODEL_SHA=sha256(CKPT)
SPLIT_SHA=sha256(SOURCE_V4_ROOT/'split_manifest.json')
H=W=2048
print('Checkpoint SHA256:',MODEL_SHA)
print('Split:',{k:len(v) for k,v in SPLIT.items()},'train observation versions:',sum(map(len,OBS.values())))
print('TEST:',len(TEST))

## 2. Native image tiling and RLE geometry

The old model saw the entire 2048×2048 observation resized to 1024×1024; these new views cover **1280 native pixels** at a **1024-pixel network input**, retaining greater effective spatial detail. Each predicted crop mask is placed back into the original 2048 coordinate system. Internal tile-boundary-truncated instances can be excluded in controlled ablations.

In [ ]:
def windows(side=TILE_SIDE, scheme=TILES):
    assert 0<side<=H
    positions=[(0,0),(W-side,0),(0,H-side),(W-side,H-side)]  # x, y
    if scheme=='FIVE':positions.append(((W-side)//2,(H-side)//2))
    if scheme not in ('FOUR','FIVE'):raise ValueError(scheme)
    return sorted(set(positions))
TILE_WINDOWS=windows()
assert all(0<=x<=W-TILE_SIDE and 0<=y<=H-TILE_SIDE for x,y in TILE_WINDOWS)

def to_rle(x):
    if isinstance(x,dict):
        return {'size':[H,W],'counts':x['counts'].encode('ascii') if isinstance(x['counts'],str) else x['counts']}
    return {'size':[H,W],'counts':x.encode('ascii') if isinstance(x,str) else x}
def to_ascii(r):
    c=r['counts'];return c.decode('ascii') if isinstance(c,bytes) else str(c)
def encode(m):return coco.encode(np.asfortranarray(np.asarray(m,dtype=np.uint8)))

def tile_mask_to_full(tile_mask,x,y,side=TILE_SIDE):
    tile_mask=np.asarray(tile_mask,dtype=np.uint8)
    if tile_mask.shape!=(side,side):tile_mask=cv2.resize(tile_mask,(side,side),interpolation=cv2.INTER_NEAREST)
    full=np.zeros((H,W),np.uint8)
    full[y:y+side,x:x+side]=tile_mask
    return full

def has_internal_tile_edge(mask,x,y,margin=12):
    if not margin or not np.any(mask):return False
    ys,xs=np.nonzero(mask)
    return bool((x>0 and xs.min()<margin) or (x+TILE_SIDE<W and xs.max()>=TILE_SIDE-margin) or
                (y>0 and ys.min()<margin) or (y+TILE_SIDE<H and ys.max()>=TILE_SIDE-margin))

# Tests: identity and exact placement near both upper and lower image boundaries
small=np.zeros((TILE_SIDE,TILE_SIDE),np.uint8);small[22:29,55:66]=1
for x,y in TILE_WINDOWS:
    full=tile_mask_to_full(small,x,y)
    assert full.shape==(H,W) and int(full.sum())==int(small.sum())
    assert int(full[y+22:y+29,x+55:x+66].sum())==77
    assert coco.area(encode(full))==77
print('Native tile windows:',TILE_WINDOWS,'coordinate unit tests passed')

## 3. CPU inference with restart-safe caches

The existing **full-disk V4 cache** can be reused when checkpoint SHA256 and inference settings match. New tile detections are always generated by YOLO from the image itself. They are saved as compressed COCO RLE plus confidence and tile coordinates. Attach prior V7 output in a subsequent Kaggle session via `PREVIOUS_V7_ROOT` to avoid recomputing expensive CPU tiles.

In [ ]:
CONFIG={'checkpoint_sha256':MODEL_SHA,'split_sha256':SPLIT_SHA,'global_imgsz':GLOBAL_IMGSZ,
        'raw_conf':RAW_CONF,'raw_iou':RAW_IOU,'tile_imgsz':TILE_IMGSZ,'tile_side':TILE_SIDE,
        'tile_scheme':TILES,'model':'YOLO11n-seg','version':'v7-native-tiles-1'}
(OUT/'v7_config.json').write_text(json.dumps(CONFIG,indent=2))
PREV=Path(PREVIOUS_V7_ROOT) if PREVIOUS_V7_ROOT else None
if PREV:
    assert (PREV/'v7_config.json').is_file(), 'Attach previous V7 run output folder'
    assert json.loads((PREV/'v7_config.json').read_text())==CONFIG,'Previous V7 cache belongs to another config or checkpoint'

G_DIR=OUT/'global';T_DIR=OUT/'tiles';G_DIR.mkdir(exist_ok=True);T_DIR.mkdir(exist_ok=True)
MODEL=None

def get_model():
    global MODEL
    if MODEL is None:
        print('Loading existing trained YOLO11n-seg on CPU:',CKPT)
        MODEL=YOLO(str(CKPT))
    return MODEL

def base_meta(kind,stem):
    return {**CONFIG,'stem':stem,'source':'LIVE_TEST_OR_TRAIN_JPEG_INFERENCE','kind':kind}

def get_cache(kind,stem):
    folder='global' if kind=='global' else 'tiles'
    paths=[OUT/folder/(stem+'.json')]
    if PREV:paths.append(PREV/folder/(stem+'.json'))
    if kind=='global':paths.append(SOURCE_V4_ROOT/'cache'/'yolo11n-seg'/(stem+'.json'))
    for p in paths:
        if not p.is_file():continue
        try:
            rec=json.loads(p.read_text())
            if p==paths[-1] and kind=='global' and p.parent.parent==SOURCE_V4_ROOT/'cache':
                # Original V4 cache schema: original checkpoint, global prediction size, TTA off.
                if (rec.get('stem')==stem and rec.get('checkpoint_sha256')==MODEL_SHA
                    and rec.get('predict_imgsz')==GLOBAL_IMGSZ and rec.get('tta') is False
                    and rec.get('source')=='LIVE_IMAGE_INFERENCE'):
                    return rec['proposals']
            else:
                if all(rec.get(k)==v for k,v in base_meta(kind,stem).items()):return rec['proposals']
        except Exception:continue
    return None

@torch.inference_mode()
def infer_view(gray,kind,x=0,y=0):
    assert gray.ndim==2
    inp=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR)
    shape=gray.shape
    imgsz=GLOBAL_IMGSZ if kind=='global' else TILE_IMGSZ
    result=get_model().predict(source=inp,imgsz=imgsz,conf=RAW_CONF,iou=RAW_IOU,
        retina_masks=True,agnostic_nms=True,max_det=MAX_DET,device='cpu',verbose=False)[0]
    if result.boxes is None or result.masks is None:return []
    masks=result.masks.data.detach().cpu().numpy()
    scores=result.boxes.conf.detach().cpu().numpy()
    assert len(scores)==len(masks)
    result_list=[]
    for score,mask in zip(scores,masks):
        local=(mask>.5).astype(np.uint8)
        if local.shape!=shape:local=cv2.resize(local,(shape[1],shape[0]),interpolation=cv2.INTER_NEAREST)
        if not local.any():continue
        truncated=bool(has_internal_tile_edge(local,x,y)) if kind=='tile' else False
        full=local if kind=='global' else tile_mask_to_full(local,x,y)
        rr=encode(full)
        result_list.append({'counts':to_ascii(rr),'score':float(score),'area':int(coco.area(rr)),
                            'origin':kind,'x':int(x),'y':int(y),'truncated':truncated})
    return result_list

def infer_cached(kind,stem,image_path):
    already=get_cache(kind,stem)
    if already is not None:return already
    gray=cv2.imread(str(image_path),cv2.IMREAD_GRAYSCALE)
    if gray is None or gray.shape!=(H,W):raise RuntimeError('Image missing/shape mismatch: '+str(image_path))
    if kind=='global':found=infer_view(gray,'global')
    else:
        found=[]
        for x,y in TILE_WINDOWS:
            for p in infer_view(gray[y:y+TILE_SIDE,x:x+TILE_SIDE],'tile',x,y):found.append(p)
    rec={**base_meta(kind,stem),'proposals':found}
    dest=(G_DIR if kind=='global' else T_DIR)/(stem+'.json')
    dest.write_text(json.dumps(rec,separators=(',',':')))
    return found

def ensure(kind,stems,lookup):
    pending=[s for s in stems if get_cache(kind,s) is None]
    print(kind,'present:',len(stems)-len(pending),'pending:',len(pending))
    for stem in tqdm(pending,desc='CPU native-image '+kind+' predictions'):
        infer_cached(kind,stem,lookup[stem])

def pool(stem,mode):
    a=get_cache('global',stem)
    if a is None:raise FileNotFoundError('Global predictions absent: '+stem)
    a=[dict(p,origin='global',truncated=False) for p in a]
    if mode=='global':return a
    b=get_cache('tile',stem)
    if b is None:raise FileNotFoundError('Tile predictions absent: '+stem)
    return a+[dict(p,origin='tile',truncated=bool(p.get('truncated',False))) for p in b]

## 4. Exact original V4 control, then a genuinely new multiscale selection policy

**Important correction to V6:** confidence and area filters are applied **before** allocating exclusive pixels. Otherwise, a proposal that will eventually be discarded can erase part of another instance. The exact V4 output is also scored independently as a control (with its original filtering and ordering). New proposals are ranked by confidence and source preference, then deduplicated by both IoU and high containment. This explicitly addresses tile-edge duplicates.

In [ ]:
def original_v4(stem):
    original=get_cache('global',stem)
    if original is None:raise FileNotFoundError(stem)
    entries=sorted([p for p in original if float(p['score'])>=.32 and int(p['area'])>=350],
                   key=lambda p:(-float(p['score']),-int(p['area'])))
    occupied=np.zeros((H,W),np.uint8);prior=[];out=[]
    for p in entries:
        r=to_rle(p['counts'])
        if prior:
            vv=np.asarray(coco.iou([r],prior,[0]*len(prior)))[0]
            if vv.size and vv.max()>=.50:continue
        prior.append(r)
        mm=coco.decode(r).astype(bool)&(occupied==0)
        if mm.sum()<350:continue
        occupied[mm]=1
        out.append(encode(mm))
    return out

def distinct(rles):
    if len(rles)<2:return True
    inter=np.asarray(coco.iou(rles,rles,[0]*len(rles)))
    np.fill_diagonal(inter,0)
    return bool(np.all(inter==0))

def decode_candidate(stem,p):
    vals=pool(stem,p['mode'])
    bonus=p['tile_bonus'];conf=p['global_conf'];tile_conf=p['tile_conf']
    kept=[]
    for q in vals:
        is_tile=q.get('origin')=='tile'
        if q['score']<(tile_conf if is_tile else conf):continue
        if q['area']<p['min_area']:continue
        if is_tile and p['seam_guard'] and q.get('truncated',False):continue
        kept.append({**q,'rank':float(q['score'])+(bonus if is_tile else 0.)})
    kept.sort(key=lambda q:(-q['rank'],-q['area']))
    occupied=np.zeros((H,W),np.uint8)
    previous=[];out=[]
    for q in kept:
        rr=to_rle(q['counts'])
        # Duplicate/containment check before expensive full-resolution mask decoding
        if previous:
            iou=np.asarray(coco.iou([rr],previous,[0]*len(previous)))[0]
            areas=np.asarray(coco.area(previous),dtype=np.float64)
            intersections=iou*(areas+float(q['area']))/(1+iou+1e-9)
            containment=intersections/max(1,float(q['area']))
            if np.any(iou>=p['nms_iou']) or np.any(containment>=p['containment']):continue
        mask=coco.decode(rr).astype(bool)
        free=mask&(occupied==0)
        count=int(free.sum())
        if count<p['min_area'] or count/max(1,float(q['area']))<p['min_free_fraction']:continue
        # Crucial: update the occupied map only for a successfully kept instance.
        occupied[free]=1
        enc=encode(free)
        previous.append(rr)
        out.append(enc)
    assert distinct(out)
    return out

# Synthetic duplicate/containment tests
m1=np.zeros((H,W),np.uint8);m1[20:100,20:150]=1
m2=np.zeros((H,W),np.uint8);m2[30:70,40:80]=1
r1,r2=encode(m1),encode(m2)
assert coco.iou([r1],[r2],[0])[0,0]>0
assert distinct([r1])
assert not distinct([r1,r2])
print('Original V4 and native-tile postprocessor initialized')

## 5. Competition-style PQ proxy and failure analysis

Score each independently annotated version of each physical observation. The public competition defines PQ with matches at IoU > 0.5. This notebook uses the same matching convention as the user's V4/V6 local proxy; verify parity with the organizer's published self-evaluation before making research claims. Track TP/FP/FN, recognition (RQ), and mask fidelity (SQ).

In [ ]:
def annotation_rle(a):
    seg=a.get('segmentation')
    if isinstance(seg,dict):
        if isinstance(seg['counts'],list):return coco.frPyObjects(seg,H,W)
        return to_rle(seg['counts'])
    if not seg:return None
    if isinstance(seg[0],(int,float)):seg=[seg]
    polys=[]
    for poly in seg:
        if len(poly)<6 or len(poly)%2:continue
        pts=np.array(poly,dtype=np.float64).reshape(-1,2)
        if np.isfinite(pts).all():
            polys.append(np.clip(pts,[0,0],[W-1,H-1]).ravel().tolist())
    if not polys:return None
    ans=coco.frPyObjects(polys,H,W)
    return coco.merge(ans) if isinstance(ans,list) else ans

TRUTH_MEM={}
def truth(stem):
    if stem not in TRUTH_MEM:
        TRUTH_MEM[stem]=[]
        for annotator,items in OBS[stem]:
            TRUTH_MEM[stem].append((annotator,[r for item in items if (r:=annotation_rle(item)) is not None]))
    return TRUTH_MEM[stem]

def match_counts(gt,pred):
    if not gt and not pred:return np.zeros(4,dtype=np.float64)
    if not gt:return np.array([0,0,len(pred),0],dtype=np.float64)
    if not pred:return np.array([0,0,0,len(gt)],dtype=np.float64)
    ov=np.asarray(coco.iou(gt,pred,[0]*len(pred)),dtype=np.float64)
    matched=ov>0.5
    return np.array([ov[matched].sum(),matched.sum(),(matched.sum(axis=0)==0).sum(),
                     (matched.sum(axis=1)==0).sum()],dtype=np.float64)

def summarize(v):
    sum_iou,tp,fp,fn=map(float,v)
    denom=tp+.5*fp+.5*fn
    return {'pq':float(sum_iou/denom) if denom else 0.,'sq':float(sum_iou/tp) if tp else 0.,
            'rq':float(tp/denom) if denom else 0.,'tp':int(tp),'fp':int(fp),'fn':int(fn)}

def score(params,stems,keep_details=False):
    total=np.zeros(4,dtype=np.float64);rows=[]
    for stem in stems:
        pred=original_v4(stem) if params=='v4' else decode_candidate(stem,params)
        for version,gt in truth(stem):
            v=match_counts(gt,pred);total+=v
            if keep_details:rows.append({'stem':stem,'version':version,**summarize(v)})
    return summarize(total),pd.DataFrame(rows)

r1=encode(np.pad(np.ones((12,13),np.uint8),((10,H-22),(20,W-33))))
assert np.isclose(summarize(match_counts([r1],[r1]))['pq'],1.)
assert summarize(match_counts([r1],[]))['fn']==1
print('Multi-annotator proxy PQ unit tests passed')

## 6. Stage A: 30-image CPU screening (no test images)

`TUNE` runs a diverse but bounded search on 30 physical images sampled from the V4 *tune* group. The old V4 strategy is included as a control. No leaderboard feedback, test annotations, or embedded submission is used. It writes `screening_results.csv` and checkpoint-linked caches.

In [ ]:
rng=random.Random(SEED+81)
SAMPLE=list(sorted(SPLIT['tune']));rng.shuffle(SAMPLE)
SCREEN=sorted(SAMPLE[:min(TUNE_SAMPLE,len(SAMPLE))]);TUNE_STEMS=sorted(SPLIT['tune'])

def parameter_grid():
    # Only models with genuinely new tile proposals are compared to exact old V4.
    # A stronger tile confidence prevents indiscriminate growth of false positives.
    for gc,tc,bonus,guard,nms in product([.28,.36],[.35,.48,.62],[0.,.10], [False,True], [.50,.70]):
        yield {'mode':'tiles','global_conf':gc,'tile_conf':tc,'tile_bonus':bonus,
               'min_area':350,'seam_guard':guard,'nms_iou':nms,'containment':.88,
               'min_free_fraction':.50}
    # Proposed crop-free new global control (changes filtering order, not detector)
    yield {'mode':'global','global_conf':.32,'tile_conf':1.,'tile_bonus':0.,
           'min_area':350,'seam_guard':False,'nms_iou':.50,'containment':.88,
           'min_free_fraction':.50}
PARAMS=list(parameter_grid())
SCREEN_RESULTS=OUT/'screening_results.csv'

def run_screen():
    ensure('global',SCREEN,TRAIN);ensure('tile',SCREEN,TRAIN)
    table=[]
    control,_=score('v4',SCREEN)
    table.append({'index':-1,'mode':'original_v4',**control})
    for i,p in enumerate(tqdm(PARAMS,desc='Tile PQ screening')):
        sc,_=score(p,SCREEN)
        table.append({'index':i,**p,**sc})
    board=pd.DataFrame(table).sort_values(['pq','rq'],ascending=False)
    board.to_csv(SCREEN_RESULTS,index=False)
    print('Screening V4 score:',control['pq'])
    display(board.head(12))
    return board

if MODE in ('TUNE','ALL'):
    board=run_screen()
else:
    if not SCREEN_RESULTS.exists():
        if PREV and (PREV/'screening_results.csv').exists():board=pd.read_csv(PREV/'screening_results.csv')
        else:raise RuntimeError('Run MODE=TUNE and save/reattach output before CONFIRM/SUBMIT')
    else:board=pd.read_csv(SCREEN_RESULTS)

## 7. Stage B: full 97-image tuning and previously seen audit regression test

Only the best screening candidates get evaluated on *all* V4 tuning images. **Do not declare success based solely on the 30-image screening PQ.** The original audit was examined in earlier runs, so it is reported only as a regression check. The selected recipe is hashed and frozen in `selection.json`. A model is promoted only if full tuning gain ≥ `MIN_GAIN_TO_PROMOTE`; otherwise the original V4 control wins.

In [ ]:
CONFIRM_RESULTS=OUT/'full_tune_results.csv'
SELECTION=OUT/'selection.json'
AUDIT_REPORT=OUT/'audit_comparison.json'

def run_confirm():
    ensure('global',TUNE_STEMS,TRAIN);ensure('tile',TUNE_STEMS,TRAIN)
    ranked=[int(i) for i in board['index'] if int(i)>=0][:CONFIRM_TOP]
    # Include global-only candidate to detect an apparent gain from filtering rather than tile information.
    global_ids=[i for i,p in enumerate(PARAMS) if p['mode']=='global']
    ranked=list(dict.fromkeys(ranked+global_ids))
    v4,_=score('v4',TUNE_STEMS)
    rows=[{'index':-1,'mode':'original_v4',**v4}]
    for i in tqdm(ranked,desc='Full tune on 97 physical observations'):
        ss,_=score(PARAMS[i],TUNE_STEMS)
        rows.append({'index':i,**PARAMS[i],**ss})
    comparison=pd.DataFrame(rows).sort_values(['pq','rq'],ascending=False)
    comparison.to_csv(CONFIRM_RESULTS,index=False)
    top=comparison.iloc[0]
    gain=float(top['pq']-v4['pq'])
    chosen='v4' if (int(top['index'])<0 or gain<MIN_GAIN_TO_PROMOTE) else PARAMS[int(top['index'])]
    choice={'strategy':chosen,'full_tune_pq':float(top['pq']),'original_v4_tune_pq':v4['pq'],
            'gain':gain,'min_gain_to_promote':MIN_GAIN_TO_PROMOTE,
            'source_model_sha256':MODEL_SHA,'split_sha256':SPLIT_SHA,'config':CONFIG,
            'selection':'full 97-image grouped tune only; previous audit used for regression reporting only'}
    SELECTION.write_text(json.dumps(choice,indent=2))
    print('FULL TUNE RESULT:');display(comparison.head(12));print('CHOSEN:',choice['strategy'],'GAIN',gain)
    if AUDIT_PREVIOUSLY_SEEN:
        audit=sorted(SPLIT['audit'])
        ensure('global',audit,TRAIN)
        if chosen!='v4':ensure('tile',audit,TRAIN)
        old,old_details=score('v4',audit,True)
        new,new_details=score(chosen,audit,True)
        report={'status':'PREVIOUSLY_SEEN_AUDIT_NOT_INDEPENDENT','original_v4':old,'selected':new,
                'delta_pq':new['pq']-old['pq'],'config':choice}
        AUDIT_REPORT.write_text(json.dumps(report,indent=2))
        new_details.to_csv(OUT/'audit_per_annotation.csv',index=False)
        print('Prior audit cross-check:',json.dumps({'old':old,'candidate':new,'delta_pq':report['delta_pq']},indent=2))
    return choice

if MODE in ('CONFIRM','ALL'):
    selection=run_confirm()
else:
    if SELECTION.exists():selection=json.loads(SELECTION.read_text())
    elif PREV and (PREV/'selection.json').exists():selection=json.loads((PREV/'selection.json').read_text())
    else:selection=None
if selection is not None:
    assert selection['source_model_sha256']==MODEL_SHA and selection['split_sha256']==SPLIT_SHA
    assert selection['config']==CONFIG

## 8. Final test inference and strict submission export

Run `MODE='SUBMIT'` **after** full-group `CONFIRM`, attaching V7 output from earlier sessions if needed. If the tile method failed the promotion gate, exports the reproducible V4 control instead. Submission contains one positive-area disjoint RLE per predicted filament, every ID from the official test images only. There are no dummy masks.

In [ ]:
SUB_PATH=Path('/kaggle/working/submission.csv') if Path('/kaggle/working').exists() else OUT/'submission.csv'

def make_submission():
    if selection is None:
        raise RuntimeError('No selection from full tuning. Run MODE=CONFIRM, then MODE=SUBMIT, or MODE=ALL.')
    chosen=selection['strategy']
    ensure('global',sorted(TEST),TEST)
    if chosen!='v4':ensure('tile',sorted(TEST),TEST)
    rows=[];summ=[]
    for stem in tqdm(sorted(TEST),desc='Generating genuine full-resolution test masks'):
        rles=original_v4(stem) if chosen=='v4' else decode_candidate(stem,chosen)
        assert distinct(rles),'Predicted instance masks overlap: '+stem
        for j,rr in enumerate(rles,1):
            assert coco.area(rr)>0
            rows.append((f'{stem}_{j}',to_ascii(rr)))
        summ.append({'image':stem,'instances':len(rles)})
    frame=pd.DataFrame(rows,columns=['filament_id','segmentation_rle'])
    assert not frame.empty and frame['filament_id'].is_unique and not frame.isna().any().any()
    frame.to_csv(SUB_PATH,index=False)
    check=pd.read_csv(SUB_PATH,keep_default_na=False)
    assert list(check.columns)==['filament_id','segmentation_rle']
    for stem,group in check.groupby(check.filament_id.str.rsplit('_',n=1).str[0]):
        assert stem in TEST
        rles=[to_rle(s) for s in group['segmentation_rle']]
        assert distinct(rles)
        assert all(coco.area(r)>0 for r in rles)
    stats=pd.DataFrame(summ)
    stats.to_csv(OUT/'test_instances_per_image.csv',index=False)
    report={'source':'LIVE_IMAGE_INFERENCE','checkpoint':str(CKPT),'sha256_checkpoint':MODEL_SHA,
            'csv':str(SUB_PATH),'sha256_csv':sha256(SUB_PATH),'strategy':chosen,
            'rows':len(frame),'test_images':len(TEST),'images_with_predictions':int((stats.instances>0).sum()),
            'test_labels_used':False,'precompiled_submissions_used':False}
    (OUT/'submission_report.json').write_text(json.dumps(report,indent=2))
    print('Submission ready:',json.dumps(report,indent=2))
    return report

if MODE in ('SUBMIT','ALL'):report=make_submission()
else:print('No test inference in this mode. Next run MODE=CONFIRM then MODE=SUBMIT.')

## 9. Checkpoint resume instructions and scientific interpretation

**For CPU Kaggle:** Save the entire `/kaggle/working/solar_cpu_v7/` folder as a Kaggle output dataset and attach it to a later session. Set `PREVIOUS_V7_ROOT` to that folder under `/kaggle/input/...` and choose `MODE='CONFIRM'` or `MODE='SUBMIT'`. The notebook never silently treats an incompatible cache as evidence.

**Interpreting outcomes:** If no tile candidate improves full tuning PQ by at least 0.015, the old pipeline remains the preferred control. If tile predictions improve recall but cause more false positives, report the SQ/RQ and TP/FP/FN changes. A meaningful new method must improve the held-out audit or future grouped folds as well as Kaggle PQ. Repeated public submissions should not substitute for valid train-only model selection.

**Honest limitation:** CPU zoom-in inference cannot guarantee a new score; the fixed nano model may have learned a scale-dependent appearance that does not transfer to tiles. If V7 fails, the best next use of compute is to train a stronger pretrained segmentation detector (e.g. a medium/large model or separately trained high-resolution instance detector) using permitted labels, not tune the same checkpoint indefinitely.